# Full Merge Cleaning

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path("../../../data/merges/CS_PSEO_IPEDS_BLS_ETPL_merged.csv")

df = pd.read_csv(file_path, low_memory=False)

print("Dataset shape:", df.shape)
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

display(df.head())

Dataset shape: (26678, 245)
Total rows: 26678
Total columns: 245


,UNITID,OPEID6,INSTNM,state,city,bea_region,ZIP,MAIN,CIPCODE,CIPDESC,...,local_earnings_index_region,local_earnings_index,local_index_level,earnings_1yr,earnings_1yr_local_adj,value_added_earnings_local,program_cost,vae_positive_local,passes_vae_local,earnings_vs_local_field
0,100663.0,1052,University of Alabama at Birmingham,AL,Birmingham,Southeast,35294-0110,1,1609,"Romance Languages, Literatures, and Linguistics.",...,0.92534,0.746218,state,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100663.0,1052,University of Alabama at Birmingham,AL,Birmingham,Southeast,35294-0110,1,4100,"Science Technologies/Technicians, General.",...,0.92534,0.746218,state,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100663.0,1052,University of Alabama at Birmingham,AL,Birmingham,Southeast,35294-0110,1,5106,Dental Support Services and Allied Professions.,...,0.92534,0.746218,state,28181.0,37765.128014,14290.128014,NaN,1.0,NaN,0.989161
3,100663.0,1052,University of Alabama at Birmingham,AL,Birmingham,Southeast,35294-0110,1,5115,Mental and Social Health Services and Allied P...,...,0.92534,0.746218,state,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100663.0,1052,University of Alabama at Birmingham,AL,Birmingham,Southeast,35294-0110,1,5202,"Business Administration, Management and Operat...",...,0.92534,0.746218,state,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Checking out the data

In [8]:
df.info()

print("\nData type counts:")
print(df.dtypes.value_counts())

print("\nColumn names:")
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 26678 entries, 0 to 26677
Columns: 245 entries, UNITID to earnings_vs_local_field
dtypes: bool(57), float64(133), int64(30), object(4), str(21)
memory usage: 39.7+ MB

Data type counts:
float64    133
bool        57
int64       30
str         21
object       4
Name: count, dtype: int64

Column names:
['UNITID', 'OPEID6', 'INSTNM', 'state', 'city', 'bea_region', 'ZIP', 'MAIN', 'CIPCODE', 'CIPDESC', 'CREDLEV', 'CREDDESC', 'IPEDSCOUNT1', 'IPEDSCOUNT2', 'EARN_MDN_HI_1YR', 'EARN_COUNT_WNE_1YR', 'EARN_MDN_1YR', 'EARN_COUNT_WNE_5YR', 'EARN_MDN_5YR', 'NUMBRANCH', 'ST_FIPS', 'LATITUDE', 'LONGITUDE', 'HBCU', 'PBI', 'ANNHI', 'TRIBAL', 'AANAPII', 'HSI', 'NANTI', 'ADM_RATE', 'DISTANCEONLY', 'UGDS', 'UGDS_WHITE', 'UGDS_BLACK', 'UGDS_HISP', 'UGDS_ASIAN', 'UGDS_AIAN', 'UGDS_NHPI', 'UGDS_2MOR', 'CURROPER', 'COSTT4_A', 'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'INEXPFTE', 'AVGFACSAL', 'PFTFAC', 'PCTPELL', 'C150_L4', 'RET_FTL4', 'RET_PTL4', 'PCTFLOAN', 'OPENADMP', 'UGNONDS

## Missing Values?

In [9]:
missing_count = df.isna().sum()

missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percentage", ascending=False)

print("Columns with missing values:", len(missing_summary))
print("Columns without missing values:", df.shape[1] - len(missing_summary))

display(missing_summary)

Columns with missing values: 106
Columns without missing values: 139


,Missing Count,Missing Percentage
weeks_to_complete,25048,93.89
etpl_prog_earnings,24873,93.23
etpl_prog_employment_q2,24790,92.92
etpl_prog_completion,24575,92.12
y5_earnings_program,24435,91.59
...,...,...
UGDS_HISP,4,0.01
UGDS_ASIAN,4,0.01
UGDS_AIAN,4,0.01
UGDS_NHPI,4,0.01


## Duplicate Rows?

In [10]:
print("Exact duplicate rows:", df.duplicated().sum())

key_columns = ["UNITID", "CIPCODE", "CREDLEV"]

duplicates = df.duplicated(
    subset=key_columns,
    keep=False
)

print("Rows sharing institution-program keys:", duplicates.sum())

if duplicates.any():
    display(
        df.loc[duplicates, key_columns + ["INSTNM", "CIPDESC"]]
        .sort_values(key_columns)
        .head(20)
    )

Exact duplicate rows: 0
Rows sharing institution-program keys: 0


## Unique values -> Data Types

In [11]:
text_columns = df.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Total text columns:", len(text_columns))

for col in text_columns:
    print(f"\nColumn: {col}")
    print("Data type:", df[col].dtype)
    print("Unique values:", df[col].nunique(dropna=True))
    print("Sample values:", df[col].dropna().unique()[:10])

Total text columns: 25

Column: INSTNM
Data type: str
Unique values: 3012
Sample values: <StringArray>
[   'University of Alabama at Birmingham',
    'University of Alabama in Huntsville',
              'The University of Alabama',
      'Central Alabama Community College',
                'Athens State University',
        'Auburn University at Montgomery',
                      'Auburn University',
 'Chattahoochee Valley Community College',
     'Enterprise State Community College',
      'Coastal Alabama Community College']
Length: 10, dtype: str

Column: state
Data type: str
Unique values: 58
Sample values: <StringArray>
['AL', 'AK', 'AZ', 'NM', 'AR', 'CA', 'CO', 'CT', 'DE', 'DC']
Length: 10, dtype: str

Column: city
Data type: str
Unique values: 1419
Sample values: <StringArray>
[    'Birmingham',     'Huntsville',     'Tuscaloosa', 'Alexander City',
         'Athens',     'Montgomery',         'Auburn',    'Phenix City',
     'Enterprise',    'Bay Minette']
Length: 10, dtype: str

## Fixing Boolean Columns

In [12]:
boolean_columns = [
    "xw_has_hot_technology",
    "xw_has_in_demand",
    "etpl_prog_wp_length_ok",
    "etpl_prog_likely_wp"
]

for col in boolean_columns:
    df[col] = df[col].astype("boolean")

print(df[boolean_columns].dtypes)

print("\nMissing values:")
print(df[boolean_columns].isna().sum())

xw_has_hot_technology     boolean
xw_has_in_demand          boolean
etpl_prog_wp_length_ok    boolean
etpl_prog_likely_wp       boolean
dtype: object

Missing values:
xw_has_hot_technology       785
xw_has_in_demand            785
etpl_prog_wp_length_ok    23209
etpl_prog_likely_wp       23209
dtype: int64


## Checking out invalid values

In [13]:
nonnegative_columns = [
    "COSTT4_A",
    "TUITIONFEE_IN",
    "TUITIONFEE_OUT",
    "prog_tuition_fees",
    "total_prog_cost",
    "xw_occ_median_wage",
    "xw_occ_mean_wage",
    "earnings_1yr",
    "etpl_prog_earnings"
]

proportion_columns = [
    "PCTPELL",
    "PCTFLOAN",
    "ADM_RATE",
    "C150_L4",
    "RET_FTL4",
    "RET_PTL4"
]

for col in nonnegative_columns:
    if col in df.columns:
        print(f"{col}: {(df[col] < 0).sum()} negative values")

print("\nProportion columns outside [0, 1]:")

for col in proportion_columns:
    if col in df.columns:
        invalid = ((df[col] < 0) | (df[col] > 1)).sum()
        print(f"{col}: {invalid} invalid values")

COSTT4_A: 0 negative values
TUITIONFEE_IN: 0 negative values
TUITIONFEE_OUT: 0 negative values
prog_tuition_fees: 0 negative values
total_prog_cost: 0 negative values
xw_occ_median_wage: 0 negative values
xw_occ_mean_wage: 0 negative values
earnings_1yr: 0 negative values
etpl_prog_earnings: 0 negative values

Proportion columns outside [0, 1]:
PCTPELL: 0 invalid values
PCTFLOAN: 0 invalid values
ADM_RATE: 0 invalid values
C150_L4: 0 invalid values
RET_FTL4: 0 invalid values
RET_PTL4: 0 invalid values


## Checking text formatting and missing-value placeholders

In [14]:
text_columns = df.select_dtypes(
    include=["str", "object"]
).columns

placeholders = ["NULL", "N/A", "NA", "**", "*", "MISSING"]

for col in text_columns:
    values = df[col].dropna().astype("str")

    whitespace_count = (
        values != values.str.strip()
    ).sum()

    placeholder_count = (
        values.str.strip().str.upper().isin(placeholders)
    ).sum()

    if whitespace_count > 0 or placeholder_count > 0:
        print(f"\nColumn: {col}")
        print("Extra whitespace:", whitespace_count)
        print("Missing placeholders:", placeholder_count)

## Standardizing column names

In [15]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Duplicate column names:", df.columns.duplicated().sum())

print("Dataset shape:", df.shape)

print("\nUpdated column names:")
print(df.columns.tolist())

Duplicate column names: 0
Dataset shape: (26678, 245)

Updated column names:
['unitid', 'opeid6', 'instnm', 'state', 'city', 'bea_region', 'zip', 'main', 'cipcode', 'cipdesc', 'credlev', 'creddesc', 'ipedscount1', 'ipedscount2', 'earn_mdn_hi_1yr', 'earn_count_wne_1yr', 'earn_mdn_1yr', 'earn_count_wne_5yr', 'earn_mdn_5yr', 'numbranch', 'st_fips', 'latitude', 'longitude', 'hbcu', 'pbi', 'annhi', 'tribal', 'aanapii', 'hsi', 'nanti', 'adm_rate', 'distanceonly', 'ugds', 'ugds_white', 'ugds_black', 'ugds_hisp', 'ugds_asian', 'ugds_aian', 'ugds_nhpi', 'ugds_2mor', 'curroper', 'costt4_a', 'tuitionfee_in', 'tuitionfee_out', 'inexpfte', 'avgfacsal', 'pftfac', 'pctpell', 'c150_l4', 'ret_ftl4', 'ret_ptl4', 'pctfloan', 'openadmp', 'ugnonds', 'stufacr', 'ipedscount1_was_missing', 'ipedscount2_was_missing', 'costt4_a_was_missing', 'tuitionfee_in_was_missing', 'tuitionfee_out_was_missing', 'pctpell_was_missing', 'ugds_was_missing', 'stufacr_was_missing', 'openadmp_was_missing', 'pctfloan_was_missing',

## Correcting one standardize column name

In [16]:
df = df.rename(columns={
    "locale_3_0": "locale_neg3_0"
})

print("Corrected column:", "locale_neg3_0" in df.columns)
print("Duplicate column names:", df.columns.duplicated().sum())
print("Final shape:", df.shape)

Corrected column: True
Duplicate column names: 0
Final shape: (26678, 245)


## Checking infinite values

In [17]:
numeric_columns = df.select_dtypes(include="number")

infinite_counts = pd.Series(
    np.isinf(numeric_columns.to_numpy(dtype="float64")).sum(axis=0),
    index=numeric_columns.columns
)

infinite_counts = infinite_counts[infinite_counts > 0]

print("Columns containing infinite values:", len(infinite_counts))
print("Total infinite values:", infinite_counts.sum())

display(infinite_counts)

Columns containing infinite values: 0
Total infinite values: 0


Series([], dtype: int64)

## Checking identifier consistency

In [18]:
identifier_columns = [
    "unitid",
    "opeid6",
    "cipcode",
    "cipcode_str",
    "cip4"
]

for col in identifier_columns:
    print(f"\nColumn: {col}")
    print("Data type:", df[col].dtype)
    print("Missing values:", df[col].isna().sum())
    print("Unique values:", df[col].nunique(dropna=True))
    print("Sample values:", df[col].dropna().unique()[:10])


Column: unitid
Data type: float64
Missing values: 0
Unique values: 3075
Sample values: [100663. 100706. 100751. 100760. 100812. 100830. 100858. 101028. 101143.
 101161.]

Column: opeid6
Data type: int64
Missing values: 0
Unique values: 2584
Sample values: [ 1052  1055  1051  1007  1008  8310  1009 12182  1015  1060]

Column: cipcode
Data type: int64
Missing values: 0
Unique values: 360
Sample values: [1609 4100 5106 5115 5202 5203 5212 5214 5218 1601]

Column: cipcode_str
Data type: int64
Missing values: 0
Unique values: 360
Sample values: [1609 4100 5106 5115 5202 5203 5212 5214 5218 1601]

Column: cip4
Data type: int64
Missing values: 0
Unique values: 360
Sample values: [1609 4100 5106 5115 5202 5203 5212 5214 5218 1601]


## Standardizing identifiers and verifying CIP consistency

In [19]:
invalid_unitid = (df["unitid"] % 1 != 0).sum()
print("Non-integer UNITID values:", invalid_unitid)

cip_mismatch = (
    (df["cipcode"] != df["cipcode_str"]) |
    (df["cipcode"] != df["cip4"])
).sum()

print("CIP mismatches:", cip_mismatch)

if invalid_unitid == 0 and cip_mismatch == 0:
    df["unitid"] = df["unitid"].astype("Int64")
    df["opeid6"] = df["opeid6"].astype(str).str.zfill(6)

    df["cipcode_str"] = (
        df["cipcode"].astype(str).str.zfill(4)
    )

    df["cip4"] = (
        df["cip4"].astype(str).str.zfill(4)
    )

    print("\nIdentifier conversions completed.")
    display(df[
        ["unitid", "opeid6", "cipcode", "cipcode_str", "cip4"]
    ].head())
else:
    print("\nReview inconsistencies before converting.")

Non-integer UNITID values: 0
CIP mismatches: 0

Identifier conversions completed.


,unitid,opeid6,cipcode,cipcode_str,cip4
0,100663,001052,1609,1609,1609
1,100663,001052,4100,4100,4100
2,100663,001052,5106,5106,5106
3,100663,001052,5115,5115,5115
4,100663,001052,5202,5202,5202


## Checking logical consistency 

In [20]:
# 1. SOC matching
soc_inconsistent = (
    df["has_soc_match"] != df["xw_n_occupations"].notna()
)

# 2. ETPL outcomes
etpl_inconsistent = (
    (~df["has_etpl_program_outcomes"].fillna(False))
    & df["etpl_prog_earnings"].notna()
)

# 3. PSEO earnings reporting
pseo_inconsistent = (
    (df["y1_earnings_program_status"] == "reported")
    != df["y1_earnings_program"].notna()
)

print("SOC match inconsistencies:", soc_inconsistent.sum())
print("ETPL outcome inconsistencies:", etpl_inconsistent.sum())
print("PSEO earnings inconsistencies:", pseo_inconsistent.sum())

SOC match inconsistencies: 0
ETPL outcome inconsistencies: 0
PSEO earnings inconsistencies: 0


## Checking constant and redundant columns

In [21]:
constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=True) <= 1
]

print("Constant or all-missing columns:", len(constant_columns))
print(constant_columns)

identical_pairs = []

for i, col1 in enumerate(df.columns):
    for col2 in df.columns[i + 1:]:
        if df[col1].equals(df[col2]):
            identical_pairs.append((col1, col2))

print("\nIdentical column pairs:", len(identical_pairs))

for col1, col2 in identical_pairs:
    print(f"{col1} == {col2}")

Constant or all-missing columns: 4
['credlev', 'creddesc', 'curroper', 'xw_has_hot_technology']

Identical column pairs: 3
tuitionfee_in_was_missing == tuitionfee_out_was_missing
pctpell_was_missing == pctfloan_was_missing
cipcode_str == cip4


## Final check

In [22]:
print("Final dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate column names:", df.columns.duplicated().sum())

print("\nIdentifier formatting:")
print("OPEID6 length issues:", df["opeid6"].str.len().ne(6).sum())
print("CIP4 length issues:", df["cip4"].str.len().ne(4).sum())

print(
    "CIP mismatches:",
    (df["cipcode_str"] != df["cip4"]).sum()
)

print("\nNullable Boolean columns:")
print(df[
    [
        "xw_has_hot_technology",
        "xw_has_in_demand",
        "etpl_prog_wp_length_ok",
        "etpl_prog_likely_wp"
    ]
].dtypes)

Final dataset shape: (26678, 245)
Missing values: 1230434
Exact duplicate rows: 0
Duplicate column names: 0

Identifier formatting:
OPEID6 length issues: 0
CIP4 length issues: 0
CIP mismatches: 0

Nullable Boolean columns:
xw_has_hot_technology     boolean
xw_has_in_demand          boolean
etpl_prog_wp_length_ok    boolean
etpl_prog_likely_wp       boolean
dtype: object


In [ ]:
output_path = Path("../data/processed/Full_Merge_Cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

# Reload to verify
df_check = pd.read_csv(output_path, low_memory=False)

print("Saved to:", output_path)
print("Exported shape:", df_check.shape)
print("Missing values:", df_check.isna().sum().sum())
print("Duplicate rows:", df_check.duplicated().sum())
print("Column names match:", df.columns.equals(df_check.columns))